# Series 2.1 — Context Pruning

**Why AI Fails? — Engineering Lab**

---

> Organizations pay for **tokens**, not API calls.  
> Context pruning = send **only** what the model needs.

**Scenario:** An HDFS incident investigation — 2,000 log lines, one block failure, one question to Gemini.

**Core lesson:** Same model, same question — only the **context size** changes.


## End-to-End Scenario

**What this lab does from start to finish**

You are on-call for an HDFS incident. A block failed overnight and leadership wants a root-cause summary before the morning standup. This notebook walks through the full engineering flow — from a vague user question to a side-by-side token benchmark.

### Step-by-step flow

1. **Receive the question** — e.g. *"Investigate why HDFS block blk_-8775602795571523802 failed."*
2. **Clarify first (Layer 1)** — if the question is vague, the app asks scoping questions and spends **0 tokens** on logs until scope is known.
3. **Load evidence** — read `datasets/HDFS_2k.log` (2,000 synthetic log lines).
4. **Run Flow A — without pruning** — dump the entire log into the prompt via `build_unpruned_prompt()` → ~**71,000+** input tokens → call Gemini (or `--dry-run` simulation).
5. **Run Flow B — with pruning** — `prune.py` pipeline:
   - filter lines by block ID
   - drop unused columns
   - deduplicate repeated messages
   - cap rows (ERROR/WARN first)
   - summarize into a compact evidence block
   - `build_pruned_prompt()` → ~**200** input tokens → same Gemini call
6. **Compare results** — `benchmark.py` prints prompt tokens, latency, estimated cost, and **~99% savings** for the same model and same question.
7. **Live demo cell** — run `python series-2.1/app.py --dry-run` from this notebook to reproduce the benchmark with no API key.

### What you should observe

| Stage | Without pruning | With pruning |
|-------|-----------------|--------------|
| Evidence in prompt | All 2,000 lines | Block-scoped summary |
| Prompt tokens | ~71,000+ | ~200 |
| Answer quality | Baseline | Same question, same model |
| Engineering win | — | Deterministic pre-call filtering |

**Takeaway:** Context pruning happens **before** the LLM call. You pay for fewer tokens and get faster responses without changing the model.


## 1. The Problem

| Without pruning | With pruning |
|-----------------|--------------|
| All 2,000 log lines in the prompt | Filter to block-relevant lines only |
| ~**71,000+** prompt tokens | ~**200** prompt tokens |
| Slow, expensive | Fast, cheap |
| Same model, same question | Same model, same question |

### Why this matters in production

- Apps dump **full chat history**, **all RAG chunks**, or **entire log files** into every request
- You pay for every token on every call
- Latency scales with prompt size
- The model doesn't need 99% of what you're sending

**Expected dry-run result:**

```
WITHOUT PRUNING   →  ~71,000 prompt tokens
WITH PRUNING      →  ~200 prompt tokens
SAVINGS           →  ~99% prompt reduction
```


## 2. What is Context Pruning?

**Context pruning** is the practice of **removing irrelevant information from the prompt before it reaches the LLM**.

It is not about changing the model, the question, or the answer format. It is about **shrinking the evidence** the model reads so you pay for fewer input tokens and get faster responses.

### Definition

```
Context pruning = filter → dedupe → cap → summarize → prompt
                  (all BEFORE the LLM call)
```

The model still receives enough information to answer. You simply stop sending data it cannot use.

### What gets pruned vs what stays

| Prune (remove or shrink) | Keep (send to model) |
|--------------------------|----------------------|
| Log lines for unrelated block IDs | Lines matching the investigation scope |
| Duplicate messages across nodes | Unique diagnostic messages |
| Routine INFO noise | ERROR and WARN events |
| Full raw log dumps | Summarized counts + top messages |
| Unscoped data loaded "just in case" | Evidence tied to the user's question |

### When context pruning applies

Context pruning is the right tool when your prompt contains **large, structured runtime data**:

- Log files and observability traces
- RAG retrieval results (too many chunks retrieved)
- Tool outputs replayed on every agent step
- Long conversation history appended wholesale
- Database query results dumped as JSON

**Rule of thumb:** If the context *could* be filtered by scope, time, ID, or relevance score — prune it first.

### What context pruning is NOT

| Technique | Difference |
|-----------|------------|
| **Prompt caching** (Series 2.2) | Caching reuses **stable** instructions; pruning shrinks **changing** evidence |
| **Summarization** (Series 2.4) | Summarization compresses **conversation** over time; pruning filters **one request's** evidence |
| **Smaller model** | Pruning keeps the same model — you send less input, not a weaker brain |
| **Shorter user question** | The question stays the same; the **context around it** gets smaller |

### Why prune before the LLM call?

```
❌ Bad pattern (post-hoc):
   Send 71k tokens → hope model ignores irrelevant lines → pay full price

✅ Good pattern (pre-call pruning):
   Filter to 2 relevant lines → summarize → send ~200 tokens → pay 99% less
```

Pruning is **deterministic engineering** — filters run in code, are testable, and produce the same evidence block every time. You are not asking the model to "figure out what's relevant" from a wall of text.

### Production impact

| Metric | Effect of pruning |
|--------|-------------------|
| **Prompt tokens** | Primary win — often 90–99% reduction |
| **Latency** | Lower — less text for the model to pre-process |
| **Cost** | Lower — input tokens are billed per request |
| **Answer quality** | Often **better** — less noise, sharper focus |
| **Context window headroom** | More room for what actually matters |

### This lab's implementation

In Series 2.1, context pruning runs in `prune.py` **after** the user provides a scoped question (block ID) and **before** `build_pruned_prompt()`:

```
User question + block ID
    → load 2,000 logs
    → prune_hdfs_context()     ← CONTEXT PRUNING happens here
    → build_pruned_prompt()    ← only ~200 tokens enter the prompt
    → Gemini
```

> **Enterprise principle:** Prune at the retrieval boundary — filter data before it becomes prompt text, not after.


## 3. Repository Layout

```
why-ai-fails/
├── demo.py                    ← Entry point (delegates to series-2.1/app.py)
├── datasets/
│   └── HDFS_2k.log            ← 2,000-line sample log file
├── common/                    ← Shared across all Series 2 labs
│   ├── prompt_builder.py      ← Unpruned vs pruned prompt templates
│   ├── benchmark.py           ← Side-by-side savings report
│   ├── gemini_client.py       ← Gemini API wrapper
│   ├── token_usage.py         ← Token estimation
│   └── utils.py               ← Log loading, block ID extraction
└── series-2.1/
    ├── app.py                 ← CLI + benchmark runner
    ├── prune.py               ← 5-step pruning pipeline
    ├── README.md
    └── Series_2.1_Context_Pruning.ipynb   ← This notebook
```


## 4. Python Files in This Lab

Files under `series-2.1/` plus shared modules from `common/`:

| File | What it does |
|------|--------------|
| **`app.py`** | CLI entry point. Runs **two flows** side-by-side (with/without pruning), handles `--dry-run` and `--clarify-demo`, loads HDFS logs, calls Gemini or token estimator, prints the benchmark report. |
| **`prune.py`** | The 5-step pruning pipeline: `filter_logs_for_block()` → `keep_useful_columns()` → `deduplicate_messages()` → `limit_relevant_rows()` → `summarize_evidence()`. Orchestrated by `prune_hdfs_context()`. |

**Shared (`common/`):**

| File | What it does |
|------|--------------|
| `prompt_builder.py` | `build_unpruned_prompt()` (anti-pattern) vs `build_pruned_prompt()` (compact evidence dict). |
| `benchmark.py` | Side-by-side printer — prompt tokens, latency, savings %. |
| `gemini_client.py` | Wrapper around Gemini API for live runs. |
| `token_usage.py` | `estimate_tokens()` for dry-run ($0) token math. |
| `utils.py` | `load_hdfs_logs()`, `extract_block_id()`, `is_ambiguous_question()`. |

**Repo root:** `demo.py` delegates to `series-2.1/app.py` for convenience.


## 5. The Pruning Pipeline (`prune.py`)

Each step removes tokens that add **no diagnostic value**:

```
2,000 log lines
    │
    ▼  Step 1 — Filter by block ID          (~2 lines remain)
    ▼  Step 2 — Drop unused columns
    ▼  Step 3 — Deduplicate messages
    ▼  Step 4 — Cap rows (ERROR/WARN first)
    ▼  Step 5 — Summarize into evidence dict
    │
    ▼  ~200 tokens in final prompt
```

| Step | Function | What it does |
|------|----------|--------------|
| 1 | `filter_logs_for_block()` | Keep only rows mentioning the target block ID |
| 2 | `keep_useful_columns()` | Drop `raw` and other non-diagnostic fields |
| 3 | `deduplicate_messages()` | Remove identical messages on different nodes |
| 4 | `limit_relevant_rows()` | ERROR/WARN first, hard cap at 50 rows |
| 5 | `summarize_evidence()` | Counts + top 10 messages + one-line summary |

**Orchestrator:** `prune_hdfs_context(logs_df, question)` runs all five steps.


## 6. Three Layers of Token Savings

### Layer 1 — Clarify first (0 tokens)

Vague questions like *"Investigate the issue"* trigger scoping questions **before** any logs are loaded:

1. Which HDFS block ID should I investigate?
2. What time window should I use?
3. Are you looking for latency, errors, availability, or deployment failures?

> **Clarify first. Retrieve later.**

**How the code enforces this:**

- `common/utils.py` → `is_ambiguous_question()` checks for a block ID (`blk_…`) in the question
- If missing **and** the question sounds vague (`investigate`, `issue`, `problem`, …), `app.py` prints clarifying questions and **exits**
- No log file is read. No DataFrame is built. No prompt is constructed. **Zero billable tokens**

```bash
python demo.py --clarify-demo
# → scoping questions only, no HDFS_2k.log loaded
```

---

### Layer 2 — Prune context (the main engineering win)

**Principle:** Filter and summarize **before** calling the LLM — not after, not inside the model.

`prune.py` runs on a structured pandas DataFrame (not raw text), so each step is deterministic and testable.

#### Where pruning sits in the request path

```
User question (with block ID)
    → load_hdfs_logs()          # parse 2,000 lines once
    → prune_hdfs_context()      # shrink to evidence dict  ← Layer 2
    → build_pruned_prompt()     # embed only the summary
    → Gemini
```

The **unpruned** path skips `prune_hdfs_context()` and passes the full `raw` column straight into the prompt — that is the anti-pattern this lab exposes.

#### What each step saves (typical run on `HDFS_2k.log`)

| Step | Input → Output | Token impact |
|------|----------------|--------------|
| Filter by block ID | 2,000 lines → ~2 lines | **Largest win** — ~99.9% of log lines removed |
| Drop columns | Removes full `raw` line text | Fewer characters per remaining row |
| Deduplicate | Same message on multiple nodes → 1 row | Stops paying twice for identical evidence |
| Cap + severity sort | ERROR/WARN kept, INFO trimmed | Safety net if filter is too broad |
| Summarize | N rows → counts + top 10 messages + summary | Replaces replay with structured facts |

#### Evidence dict (what actually goes into the prompt)

After pruning, the prompt receives a **compact dict**, not raw logs:

```python
{
    "block_id": "blk_-8775602795571523802",
    "relevant_log_count": 2,
    "error_count": 1,
    "warning_count": 0,
    "top_messages": ["...", "..."],   # max 10 lines
    "summary": "Found 2 relevant lines for blk_… Components involved: …"
}
```

`build_pruned_prompt()` formats this into ~**179 tokens**.  
`build_unpruned_prompt()` embeds all 2,000 raw lines → ~**71,524 tokens**.

Same investigation task. Same Gemini model. **400× smaller input.**

#### Production parallels (why this pattern generalizes)

| This lab | Real production system |
|----------|------------------------|
| 2,000 HDFS log lines | Full chat history in every request |
| Filter by block ID | Filter by user ID, session, time window, tenant |
| Dedupe log messages | Dedupe repeated tool outputs / RAG chunks |
| Summarize evidence | Rolling summary instead of replaying 500 messages |
| Cap at 50 ERROR rows | Top-K retrieval with a hard token budget |

> **Rule:** If the model doesn't need it to answer the question, don't send it.

---

### Layer 3 — Measure (prove the savings)

**Principle:** You cannot optimize what you do not measure.  
Layer 3 makes the cost difference **visible and repeatable**.

#### What gets measured

`app.py` runs **both flows** on the same question and passes results to `common/benchmark.py`:

| Metric | What it tells you |
|--------|-------------------|
| **Prompt tokens** | Input size — **primary cost driver** (what pruning targets) |
| **Completion tokens** | Model output size (same task → roughly similar) |
| **Total tokens** | Prompt + completion — full request cost |
| **Latency (sec)** | Wall-clock time (live runs; scales with prompt size) |
| **Prompt reduction %** | `(unpruned − pruned) / unpruned × 100` |
| **Token reduction %** | Same formula on total tokens |

#### Dry-run vs live run

| Mode | Flag | API key? | What happens |
|------|------|----------|--------------|
| **Dry-run** | `--dry-run` | No | `estimate_tokens()` counts chars locally (`len(text) // 4`) — **$0** |
| **Live** | (none) | Yes | `gemini_client.generate()` returns real token counts + answer text |

Students without an API key can still run the full benchmark and present real numbers.

#### Example benchmark output

```
====================================
CONTEXT PRUNING BENCHMARK
====================================

WITHOUT PRUNING
Prompt Tokens     : 71524
Completion Tokens : 0          ← 0 in dry-run (no API call)
Total Tokens      : 71524
Latency           : 0.0 sec

WITH PRUNING
Prompt Tokens     : 179
Completion Tokens : 0
Total Tokens      : 179
Latency           : 0.0 sec

SAVINGS
Prompt Reduction  : 99.7%
Latency Reduction : 0.0%       ← meaningful in live runs
Token Reduction   : 99.7%
====================================
```

#### Why prompt tokens matter most

- Providers bill **per input token** on every request
- A 71k-token prompt on 1,000 investigations/day = **71M input tokens/day** from logs alone
- Pruning to 179 tokens = **~99.7% input cost reduction** on the evidence portion
- Completion tokens are similar either way — the win is entirely on the **context you choose to send**

#### Engineering checklist (present this slide)

- [ ] Scope the question before loading data (Layer 1)
- [ ] Filter to relevant records before prompt build (Layer 2)
- [ ] Summarize / cap / dedupe — never dump raw dumps (Layer 2)
- [ ] Benchmark pruned vs unpruned on every new context source (Layer 3)
- [ ] Track **prompt tokens** as the primary KPI, not API call count


## 7. Execution Flow

```
Parse CLI args
    │
    ├─ --clarify-demo?  ──► Print scoping questions → EXIT (0 tokens)
    │
    ├─ Question ambiguous? ──► Clarify → EXIT
    │
    └─ Load HDFS_2k.log into DataFrame
            │
            ├─ FLOW 1: WITHOUT pruning
            │     build_unpruned_prompt(question, ALL raw logs)
            │     → Gemini (or token estimate in dry-run)
            │
            └─ FLOW 2: WITH pruning
                  prune_hdfs_context() → evidence dict
                  build_pruned_prompt(question, evidence)
                  → Gemini (or token estimate in dry-run)
            │
            └─ print_benchmark() — side-by-side comparison
```

### Two prompts, same question

| Flow | Builder | Context |
|------|---------|---------|
| Without pruning | `build_unpruned_prompt()` | Entire 2,000-line log dump |
| With pruning | `build_pruned_prompt()` | Block ID, counts, top 10 messages, summary |


## 8. How to Run

From the **repo root**:

```bash
# Setup (once)
pip install -r requirements.txt
cp .env.example .env   # optional — only needed for live Gemini calls
```

| Command | What it does | API key? |
|---------|--------------|----------|
| `python demo.py --clarify-demo` | Layer 1 only — scoping questions | No |
| `python demo.py --dry-run` | Token comparison, no API call | No |
| `python demo.py` | Full benchmark with Gemini | Yes |
| `python series-2.1/app.py --dry-run` | Same as above, direct entry | No |

**Custom question** (must include a block ID):

```bash
python demo.py --question "Investigate why HDFS block blk_-8775602795571523802 failed."
```

**Default block ID:** `blk_-8775602795571523802` (present in `datasets/HDFS_2k.log`)


In [ ]:
# Live demo cell — run the dry-run benchmark ($0, no API key needed)
# Execute this cell during your presentation

import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "demo.py").exists() and (ROOT.parent / "demo.py").exists():
    ROOT = ROOT.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "demo.py"), "--dry-run"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(f"\nExit code: {result.returncode}")


## 9. Key Code Snippets

### Unpruned prompt (anti-pattern)

```python
# common/prompt_builder.py — embeds ALL raw logs
def build_unpruned_prompt(user_question, raw_logs):
    return f"""You are an HDFS incident investigation assistant.

User question:
{user_question}

Here are the logs:
{raw_logs}          # ← entire 2,000-line file

Please provide: root cause, component, evidence, next action"""
```

### Pruned prompt (best practice)

```python
# common/prompt_builder.py — compact evidence only
def build_pruned_prompt(user_question, evidence):
    return f"""You are an HDFS incident investigation assistant.

User question:
{user_question}

Pruned evidence:
- Block ID: {evidence['block_id']}
- Error count: {evidence['error_count']}
- Top relevant messages: ...
- Summary: {evidence['summary']}"""
```

### Pipeline entry point

```python
# series-2.1/prune.py
def prune_hdfs_context(logs, user_question):
    block_id = extract_block_id(user_question)
    filtered = filter_logs_for_block(logs, block_id)
    filtered = keep_useful_columns(filtered)
    filtered = deduplicate_messages(filtered)
    filtered = limit_relevant_rows(filtered)
    evidence = summarize_evidence(filtered, block_id)
    return filtered, evidence
```


## 10. Where Series 2.1 Fits

Series 2.1 is the **foundation** of the token economics stack:

| Lab | Topic | Builds on 2.1 by… |
|-----|-------|-------------------|
| **2.1** | **Context Pruning** | — shrink evidence before the prompt |
| 2.2 | Prompt Caching | Caching the static system prompt *after* evidence is pruned |
| 2.3 | RAG Chunking | Retrieving the *right* chunks, not all chunks |
| 2.4 | Conversation Summarization | Summarizing chat history instead of replaying it |
| 2.5 | Long-Term Memory | Compressing durable facts across 500 conversations |
| 2.6 | Memory Retrieval | Finding the right memory from 100k records |

---

## Takeaway

> **Prune before you prompt.**  
> Same model. Same question. Wildly different cost.

**Next lab:** [Series 2.2 — Prompt Caching](../series-2.2/) — don't re-process the same system prompt on every request.
